This notebook contains the code we used to run scArches for 10 random seeds on the tabula dataset: it is very expensive to run! We use Tabula Sapiens v1 as the reference and v2 as the target. We reccomend saving the HVG resulting subset files since they are expensive to calcualte.

We reccomend changing the paths and saving the highly variable genes object to a different path, since that is expensive to calculate.

In [2]:
import scanpy as sc
import scvi
import numpy as np
from tqdm import tqdm
import torch
import random
import scarches as sca

 captum (see https://github.com/pytorch/captum).


In [3]:
import torch
torch.cuda.is_available()

True

In [4]:
full_tabula_ad = sc.read("/lfs/local/0/yanay/new_tabula_sapiens.h5ad") # CHANGE THIS PATH TO export_data or wherever your Tabula v1+v2 file is downloaded
full_tabula_ad

/lfs/ampere5/0/yanay/env/micromamba/envs/dogma/lib/python3.11/site-packages/anndata/__init__.py:51: FutureWarning: `anndata.read` is deprecated, use `anndata.read_h5ad` instead. `ad.read` will be removed in mid 2024.
  warnings.warn(


AnnData object with n_obs × n_vars = 1194952 × 61852
    obs: 'donor', 'tissue', 'anatomical_position', 'method', 'cdna_plate', 'library_plate', 'notes', 'cdna_well', 'old_index', 'assay', 'sample_id', 'sample', 'replicate', '10X_run', '10X_barcode', 'ambient_removal', 'donor_method', 'donor_assay', 'donor_tissue', 'donor_tissue_assay', 'cell_ontology_class', 'cell_ontology_id', 'compartment', 'broad_cell_class', 'free_annotation', 'manually_annotated', 'published_2022', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ercc', 'pct_counts_ercc', '_scvi_batch', '_scvi_labels', 'scvi_leiden_donorassay_full', 'age', 'sex', 'ethnicity'
    var: 'ensembl_id', 'gene_symbol', 'genome', 'mt', 'ercc', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'mean', 'std'

# 3K HV Genes

In [5]:
location =  "/dfs/project/uce/tabula_benchmarks/scarches" # change this path to where you want to save the intermediate large anndatas to

In [9]:
full_tabula_ad.obs["donor_num"] = full_tabula_ad.obs["donor"].str.split("TSP", expand=True)[1].astype(int)
#full_tabula_ad.obs["donor_num"].value_counts()

In [10]:
full_tabula_ad.obs["cell_type"] = full_tabula_ad.obs["cell_ontology_class"]

In [11]:
condition_key = 'sample_id'
cell_type_key = 'cell_type'

In [13]:
#adata = full_tabula_ad

#adata_ref = adata[adata.obs["donor_num"] <= 15].copy()
#adata_query = adata[adata.obs["donor_num"] > 15].copy()

#sc.pp.highly_variable_genes(adata_ref, n_top_genes=3000, flavor="seurat_v3", subset=False)
#hv_genes = adata_ref.var["highly_variable"]

#source_adata = adata_ref[:, hv_genes].copy()
#target_adata = adata_query[:, hv_genes].copy()
#target_adata.obs["cell_type"] = "Unknown"

#print(f"Ref Size: {adata_ref.X.shape}")
#print(f"Query Size: {target_adata.X.shape}")

In [14]:
#target_adata.write("/lfs/local/0/yanay/tabula_3k_target.h5ad") # change this path to where you want to save the intermediate large anndatas to
#source_adata.write("/lfs/local/0/yanay/tabula_3k_source.h5ad") # change this path to where you want to save the intermediate large anndatas to


In [15]:
target_adata = sc.read("/lfs/local/0/yanay/tabula_3k_target.h5ad") # change this path to where you want to save the intermediate large anndatas to
source_adata = adata_ref = sc.read("/lfs/local/0/yanay/tabula_3k_source.h5ad") # change this path to where you want to save the intermediate large anndatas to


/lfs/ampere5/0/yanay/env/micromamba/envs/dogma/lib/python3.11/site-packages/anndata/__init__.py:51: FutureWarning: `anndata.read` is deprecated, use `anndata.read_h5ad` instead. `ad.read` will be removed in mid 2024.
  warnings.warn(
/lfs/ampere5/0/yanay/env/micromamba/envs/dogma/lib/python3.11/site-packages/anndata/__init__.py:51: FutureWarning: `anndata.read` is deprecated, use `anndata.read_h5ad` instead. `ad.read` will be removed in mid 2024.
  warnings.warn(


In [16]:
print(f"Ref Size: {adata_ref.X.shape}")
print(f"Query Size: {target_adata.X.shape}")

Ref Size: (613522, 3000)
Query Size: (581430, 3000)


In [17]:
for seed in range(10):
    print(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    
    sca.models.SCVI.setup_anndata(source_adata, batch_key=condition_key, labels_key=cell_type_key)
    vae = sca.models.SCVI(
        source_adata,
        n_layers=2,
        encode_covariates=True,
        deeply_inject_covariates=False,
        use_layer_norm="both",
        use_batch_norm="none",
    )
    vae.train()
    
    scanvae = sca.models.SCANVI.from_scvi_model(vae, unlabeled_category = "Unknown")
    scanvae.train(max_epochs=20)
    
    reference_latent = sc.AnnData(scanvae.get_latent_representation())
    reference_latent.obs["cell_type"] = source_adata.obs[cell_type_key].tolist()
    reference_latent.obs["batch"] = source_adata.obs[condition_key].tolist()
    
    ref_path = 'ref_model/'
    scanvae.save(ref_path, overwrite=True)
    
    model = sca.models.SCANVI.load_query_data(
        target_adata,
        ref_path,
        freeze_dropout = True,
    )
    model._unlabeled_indices = np.arange(target_adata.n_obs)
    model._labeled_indices = []
    model.train(
        max_epochs=100,
        plan_kwargs=dict(weight_decay=0.0),
        check_val_every_n_epoch=10,
    )
    
    query_latent = sc.AnnData(model.get_latent_representation())
    query_latent.obs['cell_type'] = target_adata.obs[cell_type_key].tolist()
    query_latent.obs['batch'] = target_adata.obs[condition_key].tolist()
    
    
    adata_full = source_adata.concatenate(target_adata)
    
    full_latent = sc.AnnData(model.get_latent_representation(adata=adata_full))
    full_latent.obs = adata_full.obs
    full_latent.obs['cell_type'] = adata_full.obs[cell_type_key].tolist()
    full_latent.obs['batch'] = adata_full.obs[condition_key].tolist()
    
    full_latent.write(location + f"/full_3k_seed_{seed}.h5ad")

0


/lfs/ampere5/0/yanay/env/micromamba/envs/dogma/lib/python3.11/site-packages/scvi/data/fields/_dataframe_field.py:183: UserWarning: Category 126 in adata.obs['_scvi_batch'] has fewer than 3 cells. Models may not train properly.
  categorical_mapping = _make_column_categorical(
/lfs/ampere5/0/yanay/env/micromamba/envs/dogma/lib/python3.11/site-packages/scvi/data/fields/_dataframe_field.py:183: UserWarning: Category 125 in adata.obs['_scvi_labels'] has fewer than 3 cells. Models may not train properly.
  categorical_mapping = _make_column_categorical(
Trainer will use only 1 of 8 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=8)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using

Epoch 13/13: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 13/13 [06:37<00:00, 29.86s/it, v_num=1, train_loss_step=255, train_loss_epoch=253]

`Trainer.fit` stopped: `max_epochs=13` reached.


Epoch 13/13: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 13/13 [06:37<00:00, 30.55s/it, v_num=1, train_loss_step=255, train_loss_epoch=253]


...

`Trainer.fit` stopped: `max_epochs=100` reached.


Epoch 100/100: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [1:48:10<00:00, 64.91s/it, v_num=1, train_loss_step=255, train_loss_epoch=244]


/tmp/user/21290/ipykernel_3119568/1130229504.py:46: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_full = source_adata.concatenate(target_adata)


INFO     Input AnnData not setup with scvi-tools. attempting to transfer AnnData setup                             


/lfs/ampere5/0/yanay/env/micromamba/envs/dogma/lib/python3.11/site-packages/scvi/data/fields/_dataframe_field.py:221: UserWarning: Category 126 in adata.obs['_scvi_batch'] has fewer than 3 cells. Models may not train properly.
  new_mapping = _make_column_categorical(
/lfs/ampere5/0/yanay/env/micromamba/envs/dogma/lib/python3.11/site-packages/scvi/data/fields/_dataframe_field.py:221: UserWarning: Category 125 in adata.obs['_scvi_labels'] has fewer than 3 cells. Models may not train properly.
  new_mapping = _make_column_categorical(
/lfs/ampere5/0/yanay/env/micromamba/envs/dogma/lib/python3.11/site-packages/scvi/data/fields/_scanvi.py:56: UserWarning: Category 125 in adata.obs['_scvi_labels'] has fewer than 3 cells. Models may not train properly.
  mapping = _make_column_categorical(
